# 02 · Build all comma2k19 processed data

Resume-safe: existing video + metadata pairs are skipped.

In [1]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import subprocess
import sys

# ============================================================
# Repository
# ============================================================
REPO = Path("/content/Blackbox-Detection")
REPO_URL = "https://github.com/sangchun1/Blackbox-Detection.git"
BRANCH = "stage3-sangchun"

if not REPO.exists():
    subprocess.run(
        [
            "git", "clone",
            "--branch", BRANCH,
            "--single-branch",
            REPO_URL,
            str(REPO),
        ],
        check=True,
    )
else:
    subprocess.run(
        ["git", "-C", str(REPO), "fetch", "origin", BRANCH],
        check=True,
    )
    subprocess.run(
        ["git", "-C", str(REPO), "checkout", BRANCH],
        check=True,
    )
    subprocess.run(
        ["git", "-C", str(REPO), "pull", "--ff-only", "origin", BRANCH],
        check=True,
    )

# ============================================================
# Paths
# ============================================================
DRIVE_ROOT = Path("/content/drive/MyDrive/Blackbox-Detection")
DATA_ROOT = DRIVE_ROOT / "DATASET"

COMMA_ROOT = DATA_ROOT / "comma2k19"
RAW_ROOT = COMMA_ROOT / "raw"
PROCESSED_ROOT = COMMA_ROOT / "processed" / "v1"

MANIFEST_ROOT = DRIVE_ROOT / "manifests" / "stage3" / "v1"
OUTPUT_ROOT = DRIVE_ROOT / "outputs" / "stage3"
PRETRAINED_ROOT = DRIVE_ROOT / "pretrained"

for p in [PROCESSED_ROOT, MANIFEST_ROOT, OUTPUT_ROOT, PRETRAINED_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

# ============================================================
# Install project from the existing pyproject.toml
# --no-deps keeps Colab's/DACON's binary stack intact.
# ============================================================
subprocess.run(
    [
        sys.executable,
        "-m", "pip", "install",
        "-q", "--no-deps", "-e", str(REPO),
    ],
    check=True,
)

if str(REPO / "src") not in sys.path:
    sys.path.insert(0, str(REPO / "src"))

commit = subprocess.check_output(
    ["git", "-C", str(REPO), "rev-parse", "--short", "HEAD"],
    text=True,
).strip()

print("Repository     :", REPO)
print("Branch         :", BRANCH)
print("Commit         :", commit)
print("RAW_ROOT       :", RAW_ROOT)
print("PROCESSED_ROOT :", PROCESSED_ROOT)

Mounted at /content/drive
Repository     : /content/Blackbox-Detection
Branch         : stage3-sangchun
Commit         : 5a4c7d8
RAW_ROOT       : /content/drive/MyDrive/Blackbox-Detection/DATASET/comma2k19/raw
PROCESSED_ROOT : /content/drive/MyDrive/Blackbox-Detection/DATASET/comma2k19/processed/v1


In [2]:
from pathlib import Path
import numpy as np

bad_npz = []

for path in sorted((PROCESSED_ROOT / "metadata").rglob("*.npz")):
    try:
        with np.load(path, allow_pickle=False) as data:
            _ = data.files
            # 실제 배열 하나까지 읽어서 ZIP payload 검증
            if data.files:
                _ = data[data.files[0]]
    except Exception as e:
        bad_npz.append((path, repr(e)))

print("corrupted metadata:", len(bad_npz))

for path, err in bad_npz:
    print(path, err)

corrupted metadata: 0


In [3]:
import pandas as pd
from blackbox_detection.stage3.comma2k19 import (
    find_archives,
    PrepareConfig,
    prepare_archive,
)
from blackbox_detection.stage3.manifest import build_segment_manifest

archives = find_archives(RAW_ROOT)
assert archives, f"No archives found under {RAW_ROOT}"

print("archives:", len(archives))
for p in archives:
    print(" -", p.name)

cfg = PrepareConfig(processed_root=PROCESSED_ROOT, overwrite=False)

all_reports = []
for i, archive in enumerate(archives, 1):
    print(f"\n[{i}/{len(archives)}] {archive.name}")
    rep = prepare_archive(archive, cfg)
    all_reports.append(rep)

    if "error" in rep.columns:
        bad = rep[rep["error"].notna()]
        if len(bad):
            print(f"errors in {archive.name}: {len(bad)}")
            display(bad)

report = pd.concat(all_reports, ignore_index=True)
report.to_csv(PROCESSED_ROOT / "prepare_report.csv", index=False)

manifest = build_segment_manifest(PROCESSED_ROOT)
manifest.to_csv(PROCESSED_ROOT / "manifest.csv", index=False)

print("\nsegments:", len(manifest))
print("frames  :", int(manifest.num_frames.sum()))
display(manifest.head())

archives: 10
 - Chunk_1.zip
 - Chunk_10.zip
 - Chunk_2.zip
 - Chunk_3.zip
 - Chunk_4.zip
 - Chunk_5.zip
 - Chunk_6.zip
 - Chunk_7.zip
 - Chunk_8.zip
 - Chunk_9.zip

[1/10] Chunk_1.zip


Chunk_1.zip:   0%|          | 0/188 [00:00<?, ?it/s]


[2/10] Chunk_10.zip


Chunk_10.zip:   0%|          | 0/210 [00:00<?, ?it/s]


[3/10] Chunk_2.zip


...

Chunk_8.zip:   0%|          | 0/208 [00:00<?, ?it/s]


[10/10] Chunk_9.zip


Chunk_9.zip:   0%|          | 0/210 [00:00<?, ?it/s]


segments: 2035
frames  : 1215419


,dataset,vehicle_id,route_id,segment_id,num_frames,duration_s,video_relpath,metadata_relpath
0,comma2k19,99c94dc769b5d96e,99c94dc769b5d96e|2018-05-01--08-13-53,17,600,59.899147,videos/99c94dc769b5d96e__2018-05-01--08-13-53/...,metadata/99c94dc769b5d96e__2018-05-01--08-13-5...
1,comma2k19,99c94dc769b5d96e,99c94dc769b5d96e|2018-05-01--08-13-53,18,600,59.899155,videos/99c94dc769b5d96e__2018-05-01--08-13-53/...,metadata/99c94dc769b5d96e__2018-05-01--08-13-5...
2,comma2k19,99c94dc769b5d96e,99c94dc769b5d96e|2018-05-01--08-13-53,19,600,59.899144,videos/99c94dc769b5d96e__2018-05-01--08-13-53/...,metadata/99c94dc769b5d96e__2018-05-01--08-13-5...
3,comma2k19,99c94dc769b5d96e,99c94dc769b5d96e|2018-05-01--08-13-53,20,600,59.899169,videos/99c94dc769b5d96e__2018-05-01--08-13-53/...,metadata/99c94dc769b5d96e__2018-05-01--08-13-5...
4,comma2k19,99c94dc769b5d96e,99c94dc769b5d96e|2018-05-01--08-13-53,21,600,59.899167,videos/99c94dc769b5d96e__2018-05-01--08-13-53/...,metadata/99c94dc769b5d96e__2018-05-01--08-13-5...


In [4]:
import shutil

usage = shutil.disk_usage("/content/drive/MyDrive")

video_gib = sum(
    p.stat().st_size
    for p in (PROCESSED_ROOT / "videos").rglob("*.mp4")
) / 2**30

metadata_gib = sum(
    p.stat().st_size
    for p in (PROCESSED_ROOT / "metadata").rglob("*.npz")
) / 2**30

print(f"Drive free          : {usage.free / 2**30:.1f} GiB")
print(f"Processed video GiB : {video_gib:.2f}")
print(f"Metadata GiB        : {metadata_gib:.2f}")

Drive free          : 178.0 GiB
Processed video GiB : 5.05
Metadata GiB        : 0.04


In [5]:
from blackbox_detection.stage3.comma2k19 import (
    find_archives,
    discover_segments,
)
from blackbox_detection.stage3.manifest import build_segment_manifest
import pandas as pd

# ------------------------------------------------------------
# 1. Raw archives / expected segments
# ------------------------------------------------------------
archives = find_archives(RAW_ROOT)

expected_segments = 0
per_archive = []

for archive in archives:
    n = len(discover_segments(archive))
    expected_segments += n
    per_archive.append((archive.name, n))

print("=== RAW ===")
print("archives          :", len(archives))
print("expected segments :", expected_segments)

for name, n in per_archive:
    print(f"{name:15s}: {n:4d}")


# ------------------------------------------------------------
# 2. Processing report
# ------------------------------------------------------------
report_path = PROCESSED_ROOT / "prepare_report.csv"

print("\n=== PREPARE REPORT ===")

if report_path.exists():
    report = pd.read_csv(report_path)

    if "error" in report.columns:
        errors = report[report["error"].notna()]
    else:
        errors = pd.DataFrame()

    print("report rows       :", len(report))
    print("errors            :", len(errors))

    if len(errors):
        display(
            errors[
                [
                    c for c in
                    ["archive", "route_id", "segment_id", "error"]
                    if c in errors.columns
                ]
            ]
        )
else:
    print("prepare_report.csv NOT FOUND")


# ------------------------------------------------------------
# 3. Actual processed manifest
# ------------------------------------------------------------
manifest = build_segment_manifest(PROCESSED_ROOT)

print("\n=== PROCESSED ===")
print("processed segments:", len(manifest))
print("total frames      :", int(manifest["num_frames"].sum()))
print("total duration hr :", manifest["duration_s"].sum() / 3600)

print("\nframes / segment")
print(manifest["num_frames"].describe())

print("\nduration / segment")
print(manifest["duration_s"].describe())


# ------------------------------------------------------------
# 4. Integrity checks
# ------------------------------------------------------------
dup = manifest.duplicated(["route_id", "segment_id"]).sum()

missing_video = 0
missing_meta = 0

for row in manifest.itertuples(index=False):
    if not (PROCESSED_ROOT / row.video_relpath).exists():
        missing_video += 1
    if not (PROCESSED_ROOT / row.metadata_relpath).exists():
        missing_meta += 1

print("\n=== INTEGRITY ===")
print("duplicate segments:", dup)
print("missing videos    :", missing_video)
print("missing metadata  :", missing_meta)

print("\n=== COMPLETENESS ===")
print(
    f"{len(manifest)} / {expected_segments} "
    f"({100 * len(manifest) / expected_segments:.2f}%)"
)

=== RAW ===
archives          : 10
expected segments : 2035
Chunk_1.zip    :  188
Chunk_10.zip   :  210
Chunk_2.zip    :  194
Chunk_3.zip    :  202
Chunk_4.zip    :  205
Chunk_5.zip    :  211
Chunk_6.zip    :  206
Chunk_7.zip    :  201
Chunk_8.zip    :  208
Chunk_9.zip    :  210

=== PREPARE REPORT ===
report rows       : 2035
errors            : 0

=== PROCESSED ===
processed segments: 2035
total frames      : 1215419
total duration hr : 33.78006720305589

frames / segment
count    2035.000000
mean      597.257494
std        23.639178
min        97.000000
25%       600.000000
50%       600.000000
75%       600.000000
max       601.000000
Name: num_frames, dtype: float64

duration / segment
count    2035.000000
mean       59.758350
std         2.243127
min         9.599883
25%        59.899153
50%        59.899160
75%        59.899172
max        59.999206
Name: duration_s, dtype: float64

=== INTEGRITY ===
duplicate segments: 0
missing videos    : 0
missing metadata  : 0

=== COMPLETEN

In [6]:
from blackbox_detection.stage3.schema import read_frame_table
import pandas as pd
import numpy as np

manifest = build_segment_manifest(PROCESSED_ROOT)

# 짧은 segment 분포
print("=== SHORT SEGMENTS ===")
for threshold in [100, 300, 500, 550]:
    n = (manifest["num_frames"] < threshold).sum()
    print(f"< {threshold:3d} frames : {n}")

short = manifest.nsmallest(20, "num_frames")[
    ["route_id", "segment_id", "num_frames", "duration_s"]
]
display(short)

# 신호 valid coverage: 모든 segment
rows = []

for r in manifest.itertuples(index=False):
    df = read_frame_table(PROCESSED_ROOT / r.metadata_relpath)

    rows.append({
        "route_id": r.route_id,
        "segment_id": r.segment_id,
        "frames": len(df),
        "valid_speed": df["valid_speed"].mean(),
        "valid_accel": df["valid_accel_from_speed"].mean(),
        "valid_steer": df["valid_steer"].mean(),
        "valid_yaw": df["valid_yaw"].mean(),
    })

coverage = pd.DataFrame(rows)

print("\n=== VALID COVERAGE ===")
print(
    coverage[
        ["valid_speed", "valid_accel", "valid_steer", "valid_yaw"]
    ].describe()
)

print("\nsegments with < 90% core coverage:")
bad = coverage[
    (coverage["valid_speed"] < 0.90)
    | (coverage["valid_accel"] < 0.90)
    | (coverage["valid_steer"] < 0.90)
]
print(len(bad))
display(bad.head(20))

=== SHORT SEGMENTS ===
< 100 frames : 1
< 300 frames : 4
< 500 frames : 9
< 550 frames : 28


,route_id,segment_id,num_frames,duration_s
1559,99c94dc769b5d96e|2018-11-15--01-02-09,3,97,9.599883
858,99c94dc769b5d96e|2018-07-20--16-42-49,0,111,10.999845
238,99c94dc769b5d96e|2018-06-15--19-55-32,32,213,21.199702
1679,b0c9d2329ad1606b|2018-07-29--12-02-42,31,215,21.399668
329,99c94dc769b5d96e|2018-06-25--20-56-44,13,340,33.899568
959,99c94dc769b5d96e|2018-08-01--14-42-27,2,357,35.599495
1407,99c94dc769b5d96e|2018-11-03--23-08-44,8,358,35.699495
1607,99c94dc769b5d96e|2018-11-16--15-02-13,8,458,45.749370
24,99c94dc769b5d96e|2018-05-01--10-47-27,35,470,46.899333
31,99c94dc769b5d96e|2018-05-02--11-32-52,9,515,51.399275



=== VALID COVERAGE ===
       valid_speed  valid_accel  valid_steer    valid_yaw
count  2035.000000  2035.000000  2035.000000  2035.000000
mean      0.998231     0.996523     0.998348     0.998368
std       0.002262     0.002291     0.000880     0.000440
min       0.936937     0.927928     0.962791     0.989691
25%       0.998333     0.996667     0.998333     0.998333
50%       0.998333     0.996667     0.998333     0.998333
75%       0.998333     0.996667     0.998333     0.998333
max       1.000000     0.996672     1.000000     1.000000

segments with < 90% core coverage:
0


,route_id,segment_id,frames,valid_speed,valid_accel,valid_steer,valid_yaw


In [7]:
import numpy as np
import pandas as pd

manifest = build_segment_manifest(PROCESSED_ROOT).copy()

# timestamp 기준 실질적인 processed sample rate
manifest["effective_hz"] = (
    (manifest["num_frames"] - 1)
    / manifest["duration_s"].replace(0, np.nan)
)

print("=== EFFECTIVE SAMPLE RATE ===")
print(manifest["effective_hz"].describe())

for lo, hi in [
    (9.9, 10.1),
    (9.8, 10.2),
    (9.5, 10.5),
    (9.0, 11.0),
]:
    ok = manifest["effective_hz"].between(lo, hi)
    print(
        f"{lo:.1f} ~ {hi:.1f} Hz : "
        f"{ok.sum():4d} / {len(manifest)} "
        f"({100 * ok.mean():.2f}%)"
    )

bad = manifest[
    ~manifest["effective_hz"].between(9.5, 10.5)
].sort_values("effective_hz")

print("\nOutside 9.5~10.5 Hz:", len(bad))

display(
    bad[
        [
            "route_id",
            "segment_id",
            "num_frames",
            "duration_s",
            "effective_hz",
        ]
    ].head(50)
)

=== EFFECTIVE SAMPLE RATE ===
count    2035.000000
mean        9.977870
std         0.129165
min         8.640655
25%        10.000139
50%        10.000140
75%        10.000141
max        10.000159
Name: effective_hz, dtype: float64
9.9 ~ 10.1 Hz : 1963 / 2035 (96.46%)
9.8 ~ 10.2 Hz : 1972 / 2035 (96.90%)
9.5 ~ 10.5 Hz : 1997 / 2035 (98.13%)
9.0 ~ 11.0 Hz : 2022 / 2035 (99.36%)

Outside 9.5~10.5 Hz: 38


,route_id,segment_id,num_frames,duration_s,effective_hz
1255,99c94dc769b5d96e|2018-10-12--20-13-54,9,519,59.949156,8.640655
1251,99c94dc769b5d96e|2018-10-12--20-13-54,5,523,59.949151,8.707379
1236,99c94dc769b5d96e|2018-10-12--20-13-54,13,523,59.849184,8.721923
1233,99c94dc769b5d96e|2018-10-12--20-13-54,10,525,59.949139,8.740743
1253,99c94dc769b5d96e|2018-10-12--20-13-54,7,526,59.949166,8.757420
1254,99c94dc769b5d96e|2018-10-12--20-13-54,8,527,59.949161,8.774101
1234,99c94dc769b5d96e|2018-10-12--20-13-54,11,530,59.899160,8.831509
1239,99c94dc769b5d96e|2018-10-12--20-13-54,16,530,59.899141,8.831512
1252,99c94dc769b5d96e|2018-10-12--20-13-54,6,532,59.899145,8.864901
1248,99c94dc769b5d96e|2018-10-12--20-13-54,25,534,59.949163,8.890866
